# A Logging Framework for Microsoft Fabric
We'll build logging from scratch, hit its limits, then let a single file do all of it.

Three things make logs answerable: a **record**, **context** to slice by, and a **model**.

## Prerequisites and libraries

This is a **Fabric Spark notebook**. Attach a **default lakehouse** before running cell 1, then run top to bottom. Everything below uses libraries that ship with the Fabric Spark runtime, nothing needs `pip install`:

**A note on paths.** Parts 1 and 2 read and write through the attached lakehouse using relative `Tables/...` paths: the naive logger keeps it to plain Spark, and the open-source framework manages its own paths off the default lakehouse. Part 3 upgrades to explicit **ABFS paths** built from the resolved workspace and lakehouse IDs, which is the production pattern: robust across schema folders (`mon/...`) and able to follow whichever lakehouse the deployment pipeline attaches in each stage. A default lakehouse stays attached throughout, because the IDs are discovered via `fabric.get_lakehouse_id()`; to run with no lakehouse attached at all you would resolve the lakehouse id by name instead.

- `pyspark`: Spark session, DataFrame API and types (runtime)
- `datetime`, `uuid`, `time`: Python standard library
- `notebookutils` / `mssparkutils`: Fabric lakehouse + workspace helpers (runtime)
- `sempy` / `sempy.fabric`: semantic-model creation, used inside `FabricLogger` (runtime)

For **Part 2** download `fabric_logging_utils.py` from the [GitHub repo](https://github.com/prathyusha-kamasani/Microsoft-Fabric-Logging-Framework) and upload it into the notebook's **Resources** folder. For **Part 3b** the helpers come from a shared `project_utils.py` packaged on the Spark **environment**.

> **Install `semantic-link-labs` before Part 2.** Run `%pip install semantic-link-labs` in a cell on its own at the very top, or add it to the attached Spark **environment**. The framework imports `sempy_labs`; `semantic-link-labs` is often not preinstalled, and if it's missing the framework runs a runtime `subprocess` `pip install` on import. In Fabric that mutates the live kernel and can silently break `print`/`display` output for the rest of the session. Pre-installing makes that a no-op, so nothing breaks mid-demo. Never use raw `subprocess` pip in a Fabric notebook: use the `%pip` magic or the Spark environment.

In [ ]:
# Standard Spark + Python only. No framework yet.
# All libraries used anywhere in this notebook are imported here so each Part runs standalone.
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, TimestampType
)
from datetime import datetime
import uuid
import time

spark = SparkSession.builder.getOrCreate()
PROJECT_NAME = "DemoProject"
print("Ready. Logging to the attached default lakehouse.")

## Part 1: Build the naive logger (DEMO 1)

The point of Part 1 is not to write good code. It is to write the code most of us have written, and to discover where it stops being useful.

### Step 1: log a single operation
The simplest possible thing: after an operation, append one row to a delta table.

In [ ]:
# The most naive version: one row, written by hand.
log_path = "Tables/naive_monitoring_log"

# One fixed schema we'll reuse when we wrap this in a function, so the appends line up.
schema = StructType([
    StructField("log_id",         StringType(),    False),
    StructField("notebook_name",  StringType(),    True),
    StructField("table_name",     StringType(),    True),
    StructField("operation_type", StringType(),    True),
    StructField("rows_before",    IntegerType(),   True),
    StructField("rows_after",     IntegerType(),   True),
    StructField("execution_time", StringType(),    True),   # stored as text: already a smell
    StructField("error_message",  StringType(),    True),
    StructField("logged_at",      TimestampType(), True),
])

row = [(
    str(uuid.uuid4()),
    "DailyETL",          # which notebook
    "sales_fact",        # which table
    "INSERT",            # what happened
    1000,                # rows before
    1500,                # rows after
    "1.0",               # execution time, seconds
    None,                # error message
    datetime.now(),      # when
)]

spark.createDataFrame(row, schema).write.format("delta").mode("append").save(log_path)
spark.read.format("delta").load(log_path).show(truncate=False)

### Step 2: wrap it in a function
Nobody writes that block every time. So we wrap it. This is usually where teams stop.

In [ ]:
def log_operation(notebook_name, table_name, operation_type, rows_after,
                  rows_before=None, execution_time=None, error_message=None):
    """Naive logger: append one row. Good for about a day."""
    row = [(
        str(uuid.uuid4()), notebook_name, table_name, operation_type,
        rows_before, rows_after, execution_time, error_message, datetime.now(),
    )]
    schema = StructType([
        StructField("log_id",         StringType(),    False),
        StructField("notebook_name",  StringType(),    True),
        StructField("table_name",     StringType(),    True),
        StructField("operation_type", StringType(),    True),
        StructField("rows_before",    IntegerType(),   True),
        StructField("rows_after",     IntegerType(),   True),
        StructField("execution_time", StringType(),    True),
        StructField("error_message",  StringType(),    True),
        StructField("logged_at",      TimestampType(), True),
    ])
    (spark.createDataFrame(row, schema)
        .write.format("delta").mode("append").save(log_path))

In [ ]:
# Call it from a fake ETL step, the way you would in a real notebook.
start = time.time()
# ... pretend we load some data here ...
time.sleep(1)
rows_loaded = 1500

log_operation(
    notebook_name="DailyETL",
    table_name="sales_fact",
    operation_type="INSERT",
    rows_before=1000,
    rows_after=rows_loaded,
    execution_time=str(round(time.time() - start, 2)),   # text, to match the naive schema
)

spark.read.format("delta").load(log_path).orderBy("logged_at").show(truncate=False)

### Step 3: now answer a real question
"How long did loads take last Tuesday, by layer?"

Watch this fall apart.

In [ ]:
# We can group by a raw timestamp, but there is no date dimension,
# no concept of "layer", no time-of-day buckets, and no model a
# non-coder could ever use. Time intelligence is painful from here.
(spark.read.format("delta").load(log_path)
    .groupBy("notebook_name")
    .count()
    .show())

# Questions we cannot answer cleanly without writing more plumbing:
#  - by calendar day / week / quarter   -> need dim_date
#  - by time of day                     -> need dim_time
#  - by medallion layer                 -> not even a column
#  - "show me, no code"                 -> need a semantic model + measures

**This is the part everyone rebuilds.** The record was easy. The dimensions, the model and the measures are the hard, boring, repeated bit. What if a single file just did all of it, from the project name?

## Part 2: Reveal the framework (DEMO 2)

Download `fabric_logging_utils.py` from the repo and upload it to the notebook's **Resources** folder. Make sure `semantic-link-labs` is already installed (see the prerequisites note) so this import doesn't trigger a runtime pip install. Then it is genuinely three lines.

In [ ]:
# Install semantic-link-labs before importing the framework (it is used for the
# semantic-model build). Pin the version: the framework calls
# generate_direct_lake_semantic_model(lakehouse_tables=...), and newer releases
# renamed that argument to 'tables', which breaks the call.
%pip install semantic-link-labs==0.8.11

In [ ]:
# Two of the three lines from the design-principle slide: import, then instantiate.
# The third line (log_operation) is two cells down. This one builds the whole stack.
from builtin.fabric_logging_utils import FabricLogger

logger = FabricLogger(PROJECT_NAME)   # builds the whole monitoring stack

### What those three lines just created
- **Lakehouse** `LH_DemoProject_Monitoring` (or the attached default lakehouse if present)
- **Fact** `monitoring_log`
- **Dimensions** `dim_date` (4 years), `dim_time` (1,440 minutes)
- **Semantic model** `SM_DemoProject_Monitoring` (Direct Lake, relationships wired)

Record, context and model: all three, from one project name.

In [ ]:
# Log an operation. Each argument answers one of the 9am questions.
logger.log_operation(
    notebook_name="DailyETL",
    table_name="sales_fact",
    operation_type="INSERT",
    rows_before=1000,        # how much was there
    rows_after=1500,         # how much moved
    execution_time=2.3,      # how long
    message="Daily load completed",
    error_message=None,      # did it fail
)

In [ ]:
# The commands you'll actually use day to day.
logger.show_recent(10)        # eyeball the last N operations
logger.get_statistics()       # summary stats
# logger.cleanup_old_logs(days_to_keep=90)   # keep the fact table tidy

### Eight measures, already in the model
Total Operations · Total Rows Changed · Average Execution Time · Error Count · Success Rate · Operations Today · Unique Tables · Unique Notebooks

The question we couldn't answer in Part 1 is now a click in the semantic model. No DAX on stage.

In [ ]:
# One known timing gotcha worth showing honestly:
# the semantic model can lag behind the table commits on first build.
# If it didn't appear, this retries it. It's expected, not a failure.
logger.create_semantic_model_when_ready()

## Part 3: The extended version (DEMO 3)

This is the version I run in real client solutions. It is the same idea, grown into a proper monitoring layer. The code below is genericised, no client specifics, but it mirrors how it works in production.

### From handy to industrial
The open-source file is the seed. In a real solution the record gets richer, logging moves onto the write path, and monitoring becomes a first-class layer beside Raw / Transformed / Curated.

In [ ]:
# The extended version starts by asking Fabric where it's running, so the SAME
# code works in Dev/Test/Prod and with or without a default lakehouse attached.
import sempy.fabric as fabric

workspace_id  = fabric.get_notebook_workspace_id()
lakehouse_id  = fabric.get_lakehouse_id()
NOTEBOOK_NAME = fabric.resolve_item_name(fabric.get_artifact_id())   # auto, never a stale hardcoded name

# Build an explicit ABFS path from the resolved IDs, instead of a relative "Tables/..." path.
# It addresses the schema-qualified mon tables reliably and follows whichever lakehouse the
# deployment pipeline attaches per stage. (A lakehouse is still attached: the id comes from
# get_lakehouse_id(). To run with none attached, resolve the lakehouse id by name instead.)
# Assumes a schema-enabled lakehouse, so writing under Tables/mon/ lands in the `mon` schema.
MON_BASE = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/mon"
print(f"notebook: {NOTEBOOK_NAME}  |  mon base: {MON_BASE}")

# A richer record. New columns exist because clients asked questions the base couldn't answer:
#   layer        -> slice by medallion layer (Raw / Transformed / Curated)
#   operation    -> finer-grained than INSERT/MERGE (e.g. write_sales_fact)
#   write_mode   -> append / overwrite / merge
#   started_at + completed_at -> real duration, not a single timestamp
# It lives in its own schema: mon.monitoring_log
extended_schema = StructType([
    StructField("log_id",           StringType(),    False),
    StructField("project_name",     StringType(),    False),
    StructField("notebook_name",    StringType(),    True),
    StructField("layer",            StringType(),    True),
    StructField("operation",        StringType(),    True),
    StructField("status",           StringType(),    True),
    StructField("target_table",     StringType(),    True),
    StructField("write_mode",       StringType(),    True),
    StructField("rows_before",      IntegerType(),   True),
    StructField("rows_after",       IntegerType(),   True),
    StructField("rows_affected",    IntegerType(),   True),
    StructField("duration_seconds", StringType(),    True),
    StructField("error_message",    StringType(),    True),
    StructField("started_at",       TimestampType(), True),
    StructField("completed_at",     TimestampType(), True),
])
print("Extended schema: base + layer, write_mode, started/completed, status.")

In [ ]:
# Logging on the write path: a write helper that logs itself.
# In production this lives in a shared NB_Utilities and every notebook calls it,
# so logging is never something you "remember" to do.

def write_table(df, path, table_name, mode="append",
                notebook_name=None, layer=None):
    rows_before = 0
    try:
        rows_before = spark.read.format("delta").load(f"{path}/{table_name}").count()
    except Exception:
        pass  # first run: table doesn't exist yet

    (df.write.format("delta").mode(mode)
        .option("overwriteSchema", "true").save(f"{path}/{table_name}"))
    row_count = df.count()
    print(f"  Wrote {table_name}: {row_count:,} rows ({mode})")

    if notebook_name and layer:
        ext_log_operation(
            notebook_name=notebook_name, layer=layer,
            operation=f"write_{table_name}", status="success",
            rows_affected=row_count, target_table=table_name,
            write_mode=mode, rows_before=rows_before, rows_after=row_count,
        )
    return row_count

In [ ]:
# The extended logger. Note: a logging failure must never mask the real error.
def ext_log_operation(operation, status, notebook_name=None, layer=None,
                      rows_affected=0, duration_seconds=0, error_message=None,
                      target_table=None, write_mode=None,
                      rows_before=None, rows_after=None):
    """Append one row to mon.monitoring_log. Swallows its own errors by design."""
    notebook_name = notebook_name or NOTEBOOK_NAME   # auto-resolved if the caller didn't pass one
    try:
        mon_path = f"{MON_BASE}/monitoring_log"   # ABFS path: no attached lakehouse needed
        now = datetime.now()
        row = [(
            str(uuid.uuid4()), PROJECT_NAME, notebook_name, layer, operation, status,
            target_table, write_mode, rows_before, rows_after, rows_affected,
            str(round(duration_seconds, 2)), error_message, now,
            now if status in ("success", "failed") else None,
        )]
        (spark.createDataFrame(row, extended_schema)
            .write.format("delta").mode("append").save(mon_path))
        mark = "OK" if status == "success" else "FAIL"
        print(f"  [log] {mark} {layer}/{operation} rows={rows_affected:,}")
    except Exception as e:
        # Never let logging failure hide the real problem.
        print(f"  [log-WARN] log_operation failed: {e}")

In [ ]:
# Plain-English errors. Spark stack traces are unreadable at 9am; translate them.
def friendly_error(e):
    msg = str(e)
    if "TABLE_OR_VIEW_NOT_FOUND" in msg:
        return "Source table not found. Check lakehouse and schema/table name."
    if "PATH_NOT_FOUND" in msg or "PathNotFound" in msg:
        return "Path not found. Upstream table likely doesn't exist yet, run the upstream notebook first."
    if "cannot resolve" in msg.lower():
        return "Column not found. Check source schema matches expected column names."
    return msg[:200]

In [ ]:
# Watermarks: the log and the load become the same system.
# get_watermark -> where did we get to last time
# set_watermark -> record this run; next run only pulls what changed.

def get_watermark(layer, table):
    try:
        df = spark.read.format("delta").load(f"{MON_BASE}/refresh_watermark")
        df.createOrReplaceTempView("_wm")
        row = spark.sql(f"""
            SELECT MAX(watermark_utc) AS w FROM _wm
            WHERE project_name = '{PROJECT_NAME}'
              AND layer = '{layer}' AND target_table = '{table}'
        """).collect()
        if row and row[0]["w"]:
            return row[0]["w"].strftime("%Y-%m-%dT%H:%M:%SZ")
    except Exception as e:
        print(f"  [watermark] first run? {e}")
    return None   # None on first run -> load everything

def set_watermark(layer, table, watermark_utc, rows_added=0):
    """Record how far this run got, so the next run only pulls newer rows."""
    wm_schema = StructType([
        StructField("project_name",  StringType(),    False),
        StructField("layer",         StringType(),    True),
        StructField("target_table",  StringType(),    True),
        StructField("watermark_utc", TimestampType(), True),   # pass a datetime, not a string
        StructField("rows_added",    IntegerType(),   True),
        StructField("updated_at",    TimestampType(), True),
    ])
    row = [(PROJECT_NAME, layer, table, watermark_utc, rows_added, datetime.now())]
    (spark.createDataFrame(row, wm_schema)
        .write.format("delta").mode("append").save(f"{MON_BASE}/refresh_watermark"))
    print(f"  [watermark] set {layer}/{table} -> {watermark_utc}")

# Usage in a raw-ingest notebook:
#   since = get_watermark("Raw", "sales") or "2020-01-01T00:00:00Z"
#   df = call_api(updated_since=since)
#   write_table(df, raw_path, "sales", mode="append", notebook_name="NB_Raw_Sales", layer="Raw")
#   set_watermark("Raw", "sales", watermark_utc=max_updated_at, rows_added=df.count())

In [ ]:
# Data quality on the same rails: log a row-count check at each layer.
# A drop or spike surfaces in the monitoring layer before it reaches a report.

def log_row_count_check(layer, table, expected_min=None):
    n = spark.read.format("delta").load(f"Tables/{table}").count()
    status = "success"
    msg = None
    if expected_min is not None and n < expected_min:
        status = "warning"
        msg = f"Row count {n:,} below expected minimum {expected_min:,}"
    ext_log_operation(
        notebook_name="NB_Validate", layer=layer,
        operation=f"rowcount_{table}", status=status,
        rows_affected=n, target_table=table, error_message=msg,
    )
    return n

### The monitoring layer
Execution logs, row counts, data quality, schema drift, plus a `refresh_watermark` table, all sitting beside Raw / Transformed / Curated. Same idea as three lines in Part 2, grown into something a whole team runs on.

**Takeaway:** start with three lines. Grow into the monitoring layer only when the questions demand it.

## Part 3b: How this is packaged for a client (the delivery craft)

This is the part that turns a clever notebook into something a client team can own. In a demo we define functions inline. In a real solution none of the code above is pasted into notebooks; it lives once and every notebook imports it.

### One library, imported everywhere
In a client solution the helpers live in a single `project_utils.py`, packaged as a **custom library on the Spark environment**. Every notebook starts with one import. There's one copy to maintain, it's version-controlled in the repo, and the deployment pipeline promotes the environment and the notebooks together, unchanged, from Dev to Test to Prod.

Three ways to ship shared code, and where I use each:
- **Notebook Resources** (`from builtin.fabric_logging_utils import ...`): the published framework, self-contained per notebook
- **Spark environment custom library** (`from project_utils import ...`): my default for client solutions, one copy for the whole workspace
- **`%run NB_Utilities`**: no environment needed, but couples notebooks and re-runs on every call

In [ ]:
# How every production notebook opens. The logger follows whichever lakehouse
# is attached, so the SAME code runs in Dev, Test and Prod with no edits.
# Variable Library drives the *sources* (endpoints, keys), not the logs.
try:
    from project_utils import (
        resolve_lakehouse, log_operation, friendly_error,
        write_table, merge_fact, get_watermark, set_watermark,
    )
    print("  project_utils loaded from the Spark environment")
except ImportError:
    # Graceful fallback: a missing helper must never take down a pipeline.
    print("  WARNING: running without project_utils. "
          "Attach the project Spark environment for logging + watermarks.")

## Appendix: Reset between rehearsals

In [ ]:
# Start completely fresh. WARNING: drops existing monitoring data.
# Never run this blind on stage.
# logger = FabricLogger(PROJECT_NAME, force_recreate=True)